In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import sys
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
from scipy.io import mmread
import pycats
from pandas.api.types import CategoricalDtype
import hotspot
import seaborn as sns

scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

palette = [
    '#533f7c',                        # LUM
    '#37888e',                        # NRP
    '#4c8d49',                        # SQM
    '#ce8d24',                        # MES
    '#aa4743',                        # NEC
]

BuGn9 = ["#f7fcfd","#e5f5f9","#ccece6","#99d8c9","#66c2a4","#41ae76","#238b45","#006d2c","#00441b"]
BuGn_new = ["#f7fcfd","#04702F","#006227","#005321","#00441B"]
BuGn_new_cmap = LinearSegmentedColormap.from_list('custom',BuGn_new)

## Load data

In [ ]:
adata_all = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")
adata = adata_all[adata_all.obs['celltype'].isin(['LUM','SQM','MES','NRP','NEC'])].copy()

fig_dir = "Reproducibility/Results/Plots/Malignant/"
os.makedirs(fig_dir, exist_ok = True)

In [ ]:
tmp_subset = 'Malignant'

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["LUM","NRP",'SQM',"MES","NEC"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

In [ ]:
# TF activity
TF_activity = pd.read_csv("Reproducibility/Results/LINGER/Primary/cell_population_TF_activity_Malignant.txt", index_col=0, sep='\t')

adata_TF = sc.AnnData(TF_activity.T.loc[adata.obs_names,:].copy(), obs=adata.obs)
adata_TF.obsm["X_umap"] = adata.obsm["X_umap"]
sc.pp.scale(adata_TF, max_value=10)

UMAP

In [ ]:
# Fig.2C  
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['celltype'],
    frameon=False,
    palette = palette,
    #legend_loc="on data",
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}Figure2C.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.S2B
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['sample'],
    frameon=False,
    #legend_loc="on data",
    legend_fontsize=7,
    wspace = 0.5,
    show = False
)
plt.savefig(f"{fig_dir}FigureS2B.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.2H
sc.set_figure_params(figsize=(3, 3)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata_TF,
    color=['GATA3','POU2F3','SNAI2'],
    color_map=solar_extra,
    ncols=10,
    vmax = "p99",
    vcenter = 0,
    frameon=False,
    wspace=0.1,
    show = False
)

plt.savefig(f"{fig_dir}Figure2H.pdf", bbox_inches='tight')
plt.close()

## Hotspot

In [ ]:
rna_adata = adata[:, adata.var.modality == "Gene Expression"].copy()
rna_adata.layers['counts'] = rna_adata.X
rna_adata.layers["counts"] = sp.csc_matrix(rna_adata.layers["counts"])
sc.pp.highly_variable_genes(rna_adata, n_top_genes=3000, flavor='seurat_v3', subset = True) 

In [ ]:
## Creating the Hotspot object
hs = hotspot.Hotspot(
    rna_adata,
    layer_key="counts",
    model='danb',
    latent_obsm_key="MultiVI_latent",
    umi_counts_obs_key="total_counts_RNA"
)

hs.create_knn_graph(weighted_graph=False, n_neighbors=30)

In [ ]:
# Compute pair-wise local correlations between these genes
hs_results = hs.compute_autocorrelations(jobs=4)
hs_genes = hs_results.loc[hs_results.FDR < 1e-30].index
local_correlations = hs.compute_local_correlations(hs_genes, jobs=16)

In [ ]:
## make modules
modules = hs.create_modules(
    min_gene_threshold=65, core_only=True, fdr_threshold=0.01
)

In [ ]:
for tmp_num in range(len(hs.modules)):
    if hs.modules[tmp_num] == 1:
        hs.modules[tmp_num] = 5
    elif hs.modules[tmp_num] == 2:
        hs.modules[tmp_num] = 7
    elif hs.modules[tmp_num] == 3:
        hs.modules[tmp_num] = 6
    elif hs.modules[tmp_num] == 4:
        hs.modules[tmp_num] = 4    
    elif hs.modules[tmp_num] == 5:
        hs.modules[tmp_num] = 4
    elif hs.modules[tmp_num] == 6:
        hs.modules[tmp_num] = 1  
    elif hs.modules[tmp_num] == 7:
        hs.modules[tmp_num] = 4  
    elif hs.modules[tmp_num] == 8:
        hs.modules[tmp_num] = 8
    elif hs.modules[tmp_num] == 9:
        hs.modules[tmp_num] = 9
    elif hs.modules[tmp_num] == 10:
        hs.modules[tmp_num] = 10
    elif hs.modules[tmp_num] == 11:
        hs.modules[tmp_num] = 2
    elif hs.modules[tmp_num] == 12:
        hs.modules[tmp_num] = 3
    elif hs.modules[tmp_num] == 13:
        hs.modules[tmp_num] = 11
    else:
        hs.modules[tmp_num] = -1        

In [ ]:
inferno_r = plt.cm.get_cmap('inferno_r', 256)
first_color_rgb = inferno_r(0)[:3]
new_colors = np.empty((256, 4))
new_colors[:128] = np.hstack([first_color_rgb, 1])
for i in range(128, 256):
    new_colors[i, :3] = inferno_r((i - 128) / 128)[:3]
    new_colors[i, 3] = 1  # Full opacity

custom_inferno_r = LinearSegmentedColormap.from_list("custom_inferno_r", new_colors)

In [ ]:
# Define the list of colors
colors = [
    '#533f7c',                        # LUM
    '#37888e',                        # NRP
    '#4c8d49',                        # SQM
    '#ce8d24',                        # MES
    '#aa4743',                        # NEC
    "#ff7721",                        # MTN
    "#FF69B4",                        # CYG
    "#f08080",                        # IFN
    "#00ced1",                        # SEC
    "#6495ED",                        # STR
    '#bcbd22',                        # pEMT
]

# Create a ListedColormap instead of LinearSegmentedColormap
cmap = ListedColormap(colors, name='custom')

In [ ]:
# Fig.2A
fig = plt.figure(figsize=(2, 2))
fig = hs.plot_local_correlations(vmin=-30, vmax=30, z_cmap=custom_inferno_r, mod_cmap=cmap)
plt.savefig(f"{fig_dir}Figure2A.png", bbox_inches='tight')
plt.close()

In [ ]:
module_scores = hs.calculate_module_scores()
module_scores.columns = [f"MP{x}" for x in module_scores.columns]

rna_adata_tmp = rna_adata.copy()
rna_adata_tmp.obs = pd.merge(left=rna_adata_tmp.obs, right=module_scores, how='left', left_index=True, right_index=True)

In [ ]:
# Fig.2B
sc.set_figure_params(figsize=(3, 3)) 
sc.settings.vector_friendly = True
sc.pl.umap(rna_adata_tmp, color=module_scores.columns, use_raw=False, ncols=4, color_map = "magma",
           vmax = "p99", show = False)
plt.savefig(f"{fig_dir}Figure2B.pdf", bbox_inches='tight')
plt.close()

In [ ]:
tmp_path = "Reproducibility/Results/Hotspot/Malignant/UC_DOGMA_Malignant_module_scores.txt"
module_scores.to_csv(tmp_path, sep='\t')

## Malignant celltype proportion barplot

In [ ]:
import sys
sys.path.append(os.path.abspath('Reproducibility/Scripts/Source/utility_functions'))
from scRNAseq_Visualization_Functions import *

In [ ]:
def filter_adata_by_group_count(adata, group_col, threshold):
    group_counts = adata.obs[group_col].value_counts()
    valid_groups = group_counts[group_counts >= threshold].index
    adata_filtered = adata[adata.obs[group_col].isin(valid_groups)].copy()   
    return adata_filtered

In [ ]:
merge = filter_adata_by_group_count(adata=adata, group_col='sample', threshold=50)
print(len(merge.obs['sample'].unique()))

In [ ]:
pts_order = ['UTUC_006','BC_028','BC_017','BC_022','BC_050','BC_024','BC_021','BC_031','BC_008','BC_019','BC_041',
             'BC_026','BC_018','UTUC_007','BC_038','UTUC_005','BC_010','BC_016','BC_023','UTUC_001','BC_037',
             'UTUC_004','BC_036','BC_020','UTUC_003','BC_013','BC_029']

In [ ]:
# palettable.cartocolors.qualitative.Prism_10.get_mpl_colormap()
dict_major_cell_type_color = {'LUM':"#5f4690",'NRP':"#339aa2",'SQM':"#4b9e4d",'MES':"#e89907",'NEC':"#c14a48"}

In [ ]:
plot_composition_simple(adata = merge, 
                        x = 'sample',
                        y = 'celltype', 
                        x_order=pts_order, 
                        y_order=["LUM","NRP","SQM","MES",'NEC'], norm=True, style='bar', 
                        bar_figsize=(8, 6), 
                        colors=list(dict_major_cell_type_color.values()), 
                        bar_width=0.6, bar_linkage=True, bar_link_alpha=0.5, 
                        heat_figsize=(5, 5), only_size=False, heat_size_scale=50, heat_marker='.', 
                        return_df=True, melt=False, return_df_annot=None)
plt.savefig(f"{fig_dir}Figure2E_1.pdf")
plt.close()

In [ ]:
plot_composition_complex(adata=merge,
                         x='sample', 
                         y='celltype', 
                         x_order=pts_order, 
                         y_order=["LUM","NRP","SQM","MES",'NEC'], 
                         x_annot= ['Histology', 'STAGE', 'Gender', 'Organ', 'Recurrence'], 
                         xticklabels=True, 
                         figsize=(10, 8), 
                         colors=None, 
                         palette=palettable.cartocolors.qualitative.Prism_10.get_mpl_colormap(),
                         heat_norm=True, bar_x_norm=False, bar_y_norm=True,
                         bar_width=0.6, bar_x_linkage=True, bar_x_link_alpha=0.5,
                         heat_size_scale=50, heat_marker='.')
plt.savefig(f"{fig_dir}Figure2E_2.pdf")
plt.close()

## Correlation between protein expression and each programs

In [ ]:
adt_denoised_counts = pd.read_csv("Reproducibility/Results/MultiVI/Malignant/UC_DOGMA_ADT_counts_denoised_Malignant.txt", sep='\t', index_col=0)

pro_adata = sc.AnnData(adt_denoised_counts.loc[adata.obs_names,:].copy(), obs=adata.obs)
sc.pp.normalize_total(pro_adata)
sc.pp.log1p(pro_adata)

In [ ]:
from sklearn import preprocessing

module_scores = pd.read_csv("Reproducibility/Results/Hotspot/Malignant/UC_DOGMA_Malignant_module_scores.txt", sep='\t', index_col=0)

x = module_scores.values #returns a numpy array
min_max_scaler = preprocessing.MinMaxScaler()
x_scaled = min_max_scaler.fit_transform(x)
module_scores_norm = pd.DataFrame(x_scaled)

module_scores_norm.index = module_scores.index
module_scores_norm.columns = module_scores.columns

In [ ]:
adt_denoised_norm = (
    pd.DataFrame(pro_adata.X, index=pro_adata.obs_names, columns=pro_adata.var_names)
      .merge(module_scores_norm, left_index=True, right_index=True, how="left")
)

In [ ]:
x, y = adt_denoised_norm["MP5"], adt_denoised_norm["surface-A0047-NCAM"]
rho, p = scipy.stats.spearmanr(x, y)
g = sns.jointplot(x=x, y=y, kind='kde', space=0, fill=True, thresh=0 ,
                  cmap=BuGn_new_cmap,marginal_kws={'color':'green'})
sns.regplot(x=x, y=y, scatter=False, ax=g.ax_joint, color="green")

# annotate on the joint axis (top-left)
g.ax_joint.text(
    0.04, 0.96,
    fr"rho = {rho:.2f}, p = {p:.2e}",
    transform=g.ax_joint.transAxes,
    ha="left", va="top",
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.8)
)
plt.savefig(f"{fig_dir}FigureS2E_1.pdf")
plt.close()

In [ ]:
x, y = adt_denoised_norm["MP8"], adt_denoised_norm["surface-A0058-HLAABC"]
rho, p = scipy.stats.spearmanr(x, y)
g = sns.jointplot(x=x, y=y, kind='kde', space=0, fill=True, thresh=0 ,
                  cmap=BuGn_new_cmap,marginal_kws={'color':'green'})
sns.regplot(x=x, y=y, scatter=False, ax=g.ax_joint, color="green")

# annotate on the joint axis (top-left)
g.ax_joint.text(
    0.04, 0.96,
    fr"rho = {rho:.2f}, p = {p:.2e}",
    transform=g.ax_joint.transAxes,
    ha="left", va="top",
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.8)
)
plt.savefig(f"{fig_dir}FigureS2E_2.pdf")
plt.close()